# Neo4j Aura Agent + Google ADK

An **Aura Agent** is a managed agent that Neo4j hosts alongside your database. You configure it in
the Aura Console - its ontology, its tools, its instructions - and Neo4j runs it. It does its own
chain-of-thought reasoning over the graph and returns an answer.

Exposing it as an **MCP server** makes that agent callable from anywhere. This notebook connects
one to Google ADK, where it becomes a tool your own agent can use.


| Feature | Description |
|---------|-------------|
| **Aura Agent** | Domain reasoning over the graph - hosted, configured in the Console |
| **Google ADK** | Orchestration - conversation state, other tools, workflows |

Contents:

1. Machine-to-machine authentication with Aura API credentials
2. Verifying the MCP endpoint before involving a model
3. Connecting the agent to ADK with `McpToolset`
4. Handling token expiry
5. Combining the hosted agent with local tools

Nothing is installed or hosted locally - the agent already runs in Aura.

---
## 1. Setup

In [ ]:
%pip install -q "google-adk>=2.0.0" requests aiohttp

In [12]:
import os
os.environ["PYDANTIC_DISABLE_PLUGINS"] = "1"

import json
import logging
import time
import warnings
from getpass import getpass

import requests

from google.genai import types
from google.adk import Agent
from google.adk.apps import App
from google.adk.plugins import ReflectAndRetryToolPlugin
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.function_tool import FunctionTool
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams

warnings.filterwarnings("ignore")
logging.getLogger("google.auth.transport").setLevel(logging.ERROR)

from dotenv import load_dotenv
load_dotenv()
print("ready")

ready


---
## 2. Configuration

Three things are needed, all from the Aura Console.

**Client credentials.** Profile menu → **Account settings** → **Client credentials** tab →
**Aura Agent & MCP** → **Create client credential**. Pick what the credential may access, then
save the generated values - they are shown once.

> These are *not* the same as Aura API keys. API keys authenticate against `api.neo4j.io` and work
> with the agent's REST endpoint; the MCP endpoint uses client credentials issued from the
> Aura Agent & MCP tab and a different token endpoint.

**MCP endpoint URL.** Enable **External access → MCP server** in the agent's configuration, then
copy the endpoint from the agent's `[…]` menu:

```
https://mcp.neo4j.io/agent?project_id=<PROJECT_ID>&agent_id=<AGENT_ID>
```

Set everything as environment variables so nothing sensitive lands in the notebook:

```bash
export AURA_MCP_CLIENT_ID=...
export AURA_MCP_CLIENT_SECRET=...
export AURA_AGENT_MCP_URL="https://mcp.neo4j.io/agent?project_id=...&agent_id=..."
```

> An externally accessible agent incurs charges per [Neo4j Aura pricing](https://neo4j.com/pricing/).
> Availability can be switched off again from the same menu.

### About the agent used here

This notebook connects to an **Investment Support Agent** built in the Neo4j Aura Console. It is
just an example - the integration below works with any Aura Agent, so use your own if you have one.

To create one:


In the [Aura Console](https://console.neo4j.io/)

Create an agent, point it at your instance, and configure its ontology, tools, and instructions.
Open **Configure → External access**, enable the **MCP server**, and update the agent.
Copy the MCP endpoint from the agent's `[…]` menu.


The [getting started tutorial](https://neo4j.com/developer/genai-ecosystem/aura-agent-getting-started/)
walks through it end to end.

Because all of the graph reasoning lives in the agent's own configuration, nothing below is
specific to this agent - only the questions in the demo cells assume investment data. Swap those
for whatever your agent knows about.

In [17]:
# Gateway token endpoint for the agent MCP server, and the fixed audience it issues for.
AURA_MCP_TOKEN_URL = "https://mcp.neo4j.io/oauth/token"
AURA_MCP_AUDIENCE = "https://agent-mcp.neo4j.io"

# Read from the environment; prompt only if missing so nothing is stored in the notebook.
CLIENT_ID = os.environ.get("AURA_MCP_CLIENT_ID") or getpass("Aura MCP client ID: ")
CLIENT_SECRET = os.environ.get("AURA_MCP_CLIENT_SECRET") or getpass("Aura MCP client secret: ")
AGENT_MCP_URL = os.environ.get("AURA_AGENT_MCP_URL") or input("Agent MCP endpoint URL: ").strip()

os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY") or getpass("Google API key: ")

GEMINI_MODEL = "gemini-flash-latest"

# Show the endpoint without leaking the IDs into notebook output.
print("Endpoint:", AGENT_MCP_URL.split("?")[0], "(project and agent IDs hidden)")

Endpoint: https://mcp.neo4j.io/agent (project and agent IDs hidden)


---
## 3. Get a token

The agent MCP server supports two authorization flows: **user (authorization code)**, which is the
browser login an MCP client like Claude Desktop or VS Code performs, and **machine-to-machine
(client credentials)**, for scripts and services. We use the second - no browser, so this runs
from a backend or a scheduled job.


> **The token endpoint allows 15 requests per hour per client ID.** Cache the token for its full
> `expires_in` window rather than fetching one per request. The helper below does that, and
> re-running the cell reuses the cached token instead of spending quota.

In [18]:
_token = {"value": None, "expires_at": 0}


def get_token(force: bool = False) -> str:
    """Return a cached bearer token, fetching a new one only when it is near expiry.

    The gateway allows 15 token requests per hour per client ID, so caching is required
    rather than merely efficient.
    """
    if not force and _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]

    response = requests.post(
        AURA_MCP_TOKEN_URL,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "audience": AURA_MCP_AUDIENCE,
        },
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    _token["value"] = payload["access_token"]
    _token["expires_at"] = time.time() + payload.get("expires_in", 3600)
    return _token["value"]


get_token()
print(f"Token cached, valid for {int(_token['expires_at'] - time.time())}s")

Token cached, valid for 86399s


---
## 4. Check the endpoint

Call the MCP endpoint directly before handing it to a model. If credentials or the URL are wrong,
the error is obvious here; inside an agent transcript it is not.

This also shows which tools the agent exposes, which is what the model will see.

In [19]:
def mcp_rpc(method: str, params: dict | None = None) -> dict:
    """Send a JSON-RPC request to the agent's MCP endpoint."""
    payload = {"jsonrpc": "2.0", "method": method, "id": 1}
    if params is not None:
        payload["params"] = params

    response = requests.post(
        AGENT_MCP_URL,
        headers={
            "Authorization": f"Bearer {get_token()}",
            "Content-Type": "application/json",
            # Streamable HTTP can answer as JSON or as a server-sent event stream.
            "Accept": "application/json, text/event-stream",
        },
        json=payload,
        timeout=60,
    )
    response.raise_for_status()

    body = response.text
    if "text/event-stream" in response.headers.get("Content-Type", ""):
        body = next(line[5:] for line in body.splitlines() if line.startswith("data:"))
    return json.loads(body)


tools = mcp_rpc("tools/list").get("result", {}).get("tools", [])
print(f"{len(tools)} tool(s) exposed by the agent:\n")
for tool in tools:
    print(f"  {tool['name']}")
    print(f"    {tool.get('description', '').splitlines()[0][:200]}")

1 tool(s) exposed by the agent:

  Investment_Support_Agent
    Assists with exploring companies, their partnerships, key personnel, and relevant articles for investment purposes.


Here I have a investment support agent which I created from neo4j console

---
## 5. Connect it to ADK

The connection is a standard `McpToolset` pointed at the remote URL with a bearer token in the
header. Nothing runs locally.

Note what the instruction does *not* say: there is no schema, no Cypher guidance, no description of
the graph. All of that lives in the Aura Agent's own configuration. The ADK agent's job is to
decide *when* to consult it and how to present the answer.

In [20]:
aura_agent_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=AGENT_MCP_URL,
        headers={"Authorization": f"Bearer {get_token()}"},
        timeout=120, 
    ),
)

app = App(
    name="aura_agent_app",
    root_agent=Agent(
        name="investment_assistant",
        model=GEMINI_MODEL,
        instruction=(
            "You help users with investment research questions.\n"
            "A hosted Neo4j Aura Agent has access to the investment knowledge graph - use its "
            "tools for anything about companies, holdings, or relationships in that data.\n"
            "Pass the user's question through clearly rather than rewriting it into keywords. "
            "Present what comes back in your own words, and say plainly if the agent could not "
            "answer rather than filling the gap from your own knowledge."
        ),
        tools=[aura_agent_tools],
    ),
    plugins=[ReflectAndRetryToolPlugin(max_retries=2)],
)

print("ADK agent connected to the Aura Agent.")

ADK agent connected to the Aura Agent.


In [21]:
async def ask(app, query, show_tools=True):
    sessions = InMemorySessionService()
    session = await sessions.create_session(state={}, app_name=app.name, user_id="user_1")
    runner = Runner(app=app, session_service=sessions)

    print(f"USER: {query}")
    answer = ""

    async for event in runner.run_async(
        session_id=session.id,
        user_id=session.user_id,
        new_message=types.Content(role="user", parts=[types.Part(text=query)]),
    ):
        if not getattr(event, "content", None) or not event.content.parts:
            continue
        for part in event.content.parts:
            if part.function_call and show_tools:
                args = json.dumps(part.function_call.args, default=str)[:150]
                print(f"   -> {part.function_call.name}({args})")
            elif part.function_response and show_tools:
                print(f"   <- {part.function_response.name}")
            elif part.text:
                answer += part.text

    print(f"\nAGENT: {answer.strip()}\n")
    return answer.strip()

In [11]:
await ask(app, "Who are the competitors of Neo4j in the graph database?")

USER: Who are the competitors of Neo4j in the graph database?


   -> Investment_Support_Agent({"query": "Who are the competitors of Neo4j in the graph database market?"})
   <- Investment_Support_Agent

AGENT: According to the investment database, the identified competitors of Neo4j include:

* **TigerGraph**
* **OrientDB**
* **Titan**
* **VESoft** (creators of NebulaGraph)
* **Dato**
* **Tunas Ridean TBK**



'According to the investment database, the identified competitors of Neo4j include:\n\n* **TigerGraph**\n* **OrientDB**\n* **Titan**\n* **VESoft** (creators of NebulaGraph)\n* **Dato**\n* **Tunas Ridean TBK**'

The tool call in the trace goes out to Aura, where the hosted agent plans, runs its own queries,
and reasons over the results. What returns is an answer, not rows - the ADK agent then decides how
to present it.

That division is the point of this integration. The graph logic lives with the graph and is
maintained in the Console; the conversational layer lives in your application. Changing the agent's
ontology or tools takes effect without touching this notebook.

---
## Summary

**Aura Agent is a hosted agent, not a database connection.** It reasons over the graph using the
ontology and tools you configured in the Console, and returns an answer rather than rows. Your ADK
agent decides when to consult it.

**Machine-to-machine authentication needs the right credentials.** Client credentials from the
**Aura Agent & MCP** tab exchange for a token at `mcp.neo4j.io/oauth/token` with audience
`https://agent-mcp.neo4j.io`. Aura API keys are a different credential for a different issuer, and
belong to the agent's REST endpoint rather than its MCP endpoint.

**Cache the token.** The gateway allows 15 token requests per hour per client ID, so hold each
token for its full lifetime. Rebuild the toolset on refresh, since `McpToolset` fixes its headers
at construction - or front the endpoint with a proxy that injects a current token per request.

**Graph logic stays with the graph.** The ADK instruction contains no schema and no Cypher.
Updating the agent's ontology or tools in the Console changes behaviour without a code change on
this side.


### Next steps

* [Aura Agent documentation](https://neo4j.com/docs/aura/aura-agent/) - configuration, external access, both authorization flows
* [Aura Agent getting started](https://neo4j.com/developer/genai-ecosystem/aura-agent-getting-started/) - building an agent from scratch
* [ADK graph workflows](https://adk.dev/graphs/) - putting the hosted agent inside a larger pipeline